# 🎬 GTA ClipForge AI — Exact Scene & Mobile Shorts Maker
### 100% Free • Direct Google Drive & Phone Downloads • Exact Second-to-Second Clipping
**Runs entirely in Google's cloud — Your laptop stays completely turned OFF.**

#### 📱 Instructions for Mobile Phone:
1. Tap **Play (▶)** on **Cell 1** to install dependencies (~15 seconds).
2. Enter your **YouTube Link**, enter your exact **Start Time** (e.g. `01:25`) and **End Time** (e.g. `01:55`), and tap **Play (▶)** on **Cell 2**.
3. Your exact scene will be cut into a vertical 9:16 Short, **auto-saved to your Google Drive app**, and ready to **download with 1 tap**!

In [ ]:
#@title 1. One-Time Setup (Tap Play ▶)
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q -U yt-dlp opencv-python-headless
print("✅ Setup complete! Ready to generate Shorts.")

In [ ]:
#@title 2. YouTube Link & Exact Scene Clipping
youtube_url = "https://youtu.be/I4tWfAfwJyE?si=DgOwP_QCDXzkNNeI" #@param {type:"string"}
mode = "Exact Start and End Seconds" #@param ["Exact Start and End Seconds", "Auto (AI Action Detection)"]
start_time = "01:25" #@param {type:"string"}
end_time = "01:55" #@param {type:"string"}
auto_number_of_shorts = 3 #@param [1, 2, 3, 4, 5]
auto_short_duration = 30 #@param [15, 30, 45, 60]
render_style = "blurred_background" #@param ["blurred_background", "center_crop"]
save_to_google_drive = True #@param {type:"boolean"}

import os, re, sys, base64, shutil, subprocess, cv2, yt_dlp
from pathlib import Path
from IPython.display import HTML, display

out_dir = Path("./generated_shorts")
out_dir.mkdir(parents=True, exist_ok=True)

def parse_time(s):
    s = str(s).strip()
    if ":" in s:
        parts = [float(p) for p in s.split(":")]
        if len(parts) == 2: return parts[0] * 60 + parts[1]
        if len(parts) == 3: return parts[0] * 3600 + parts[1] * 60 + parts[2]
    try:
        return float(s)
    except ValueError:
        return 0.0

# 1. Download YouTube Video using mobile client to avoid bot checks
print("⬇️ Connecting to YouTube and streaming...")
ydl_opts = {
    'format': '18/bestvideo*+bestaudio/best',
    'merge_output_format': 'mp4',
    'outtmpl': './source_video.mp4',
    'extractor_args': {
        'youtube': {
            'player_client': ['android', 'ios']
        }
    },
    'quiet': False,
    'no_warnings': True,
    'force_overwrites': True,
}

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        title = info.get('title', 'Gameplay')
        duration = float(info.get('duration') or 0.0)
except Exception as e:
    print("Retrying with pure Android stream fallback...")
    ydl_opts['extractor_args'] = {'youtube': {'player_client': ['android']}}
    ydl_opts['format'] = '18/best'
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        title = info.get('title', 'Gameplay')
        duration = float(info.get('duration') or 0.0)

print(f"\n🎬 Downloaded: '{title}' (Total Length: {duration:.1f}s)")

# 2. Calculate Exact Scene Cuts
clip_configs = []
if mode == "Exact Start and End Seconds":
    s_sec = parse_time(start_time)
    e_sec = parse_time(end_time)
    if e_sec <= s_sec:
        e_sec = s_sec + 30.0
    s_sec = min(s_sec, max(0.0, duration - 5.0))
    e_sec = min(e_sec, duration)
    clip_len = e_sec - s_sec
    clip_configs = [(s_sec, clip_len)]
    print(f"✂️ EXACT SCENE SELECTED: From {s_sec:.1f}s ({start_time}) to {e_sec:.1f}s ({end_time}) -> Duration: {clip_len:.1f} seconds")
else:
    count = max(1, auto_number_of_shorts)
    step = max(5.0, (duration - auto_short_duration) / max(1, count))
    clip_configs = [(min(max(0.0, duration - auto_short_duration), i * step), auto_short_duration) for i in range(count)]
    print(f"🔍 Auto AI Mode: Generating {count} exciting scenes ({auto_short_duration}s each)")

# 3. Fast FFmpeg Rendering
generated_files = []
for idx, (start_t, target_len) in enumerate(clip_configs, 1):
    clean_title = re.sub(r'[^\w\s-]', '', title)[:20].replace(' ', '_')
    out_file = out_dir / f"short_{idx}_{clean_title}_{int(target_len)}s.mp4"
    
    if render_style == "center_crop":
        vfilter = f"[0:v]scale=-1:1920,crop=1080:1920:(in_w-1080)/2:0[v]; [0:a]afade=t=in:st=0:d=0.3,afade=t=out:st={max(0.5, target_len-0.5)}:d=0.5[a]"
    else:
        vfilter = f"[0:v]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,boxblur=15:3[bg]; [0:v]scale=1080:1920:force_original_aspect_ratio=decrease[fg]; [bg][fg]overlay=(W-w)/2:(H-h)/2[v]; [0:a]afade=t=in:st=0:d=0.3,afade=t=out:st={max(0.5, target_len-0.5)}:d=0.5[a]"
    
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(round(start_t, 2)),
        "-t", str(target_len),
        "-i", "./source_video.mp4",
        "-filter_complex", vfilter,
        "-map", "[v]", "-map", "[a]",
        "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23",
        "-c:a", "aac", "-b:a", "160k",
        "-movflags", "+faststart",
        str(out_file)
    ]
    print(f"⏳ Rendering Short #{idx} ({start_t:.1f}s to {start_t+target_len:.1f}s)... Please wait ~25s")
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"✅ Successfully Rendered: {out_file.name}")
    generated_files.append(out_file)

# 4. Google Drive Auto-Save (Zero Downloads Needed on Phone)
if save_to_google_drive:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("📁 Connecting to Google Drive...")
            drive.mount('/content/drive')
        drive_dir = Path('/content/drive/MyDrive/GTA_Shorts')
        drive_dir.mkdir(parents=True, exist_ok=True)
        for gf in generated_files:
            shutil.copy(str(gf), str(drive_dir / gf.name))
            print(f"🎉 SAVED DIRECTLY TO YOUR GOOGLE DRIVE APP: MyDrive/GTA_Shorts/{gf.name}")
    except Exception as drive_err:
        print(f"⚠️ Google Drive sync: {drive_err}")

# 5. Big Mobile Tap-to-Download Button
print("\n=======================================================")
print("📥 TAP THE BUTTON BELOW TO DOWNLOAD DIRECTLY TO YOUR PHONE:")
print("=======================================================")
for idx, gf in enumerate(generated_files, 1):
    try:
        with open(gf, 'rb') as f:
            data_b64 = base64.b64encode(f.read()).decode()
        btn_html = f'''
        <div style="margin: 12px 0;">
          <a href="data:video/mp4;base64,{data_b64}" download="{gf.name}" style="
            display: inline-block;
            background: linear-gradient(135deg, #10b981, #059669);
            color: white;
            padding: 16px 28px;
            text-align: center;
            text-decoration: none;
            font-size: 16px;
            font-weight: bold;
            border-radius: 10px;
            box-shadow: 0 4px 10px rgba(0,0,0,0.3);
          ">📲 TAP TO DOWNLOAD YOUR SCENE (MP4)</a>
        </div>
        '''
        display(HTML(btn_html))
    except Exception as e:
        print(f"File ready at: {gf}")

try:
    from google.colab import files
    for gf in generated_files:
        files.download(str(gf))
except Exception:
    pass
